In [ ]:
import jupyter_client
import queue
import re
import random
import os
import json

import sys
sys.path.append("..")

# Step 2: Import the "Seed KG" (v3)
try:
    from .knowledge.vehicle_kg import TYPE_TO_DOMAIN, VALID_CONNECTIONS
    print("Successfully imported Vehicle Seed KG (v3).")
except ImportError:
    print("Error: Could not find vehicle_kg.py.")
    print("Please make sure vehicle_kg.py is in the same directory.")
    exit()

    
try:
    from .knowledge import quantity_kg
    print("Successfully imported Quantity KG (v1).")
except ImportError:
    print("Warning: Could not find quantity_kg.py.")
    print("Quantity-based heuristics will be disabled.")
    quantity_kg = None
    
# --- SysMLValidator Class (Unchanged) ---
class SysMLValidator:
    """
    Manages a connection to a SysML kernel to validate code snippets.
    """
    def __init__(self):
        print("Starting SysML kernel...")
        try:
            self.km = jupyter_client.KernelManager(kernel_name='sysml')
            self.km.start_kernel()
            self.kc = self.km.client()
            self.kc.start_channels()
            self.kc.wait_for_ready()
            print("SysML kernel started and ready!")
        except Exception as e:
            print(f"FATAL: Could not start SysML kernel. {e}")
            print("Please ensure 'sysml' is a registered Jupyter kernel.")
            self.km = None
            self.kc = None

    def validate_code(self, code):
        """Executes SysML code and captures the output and any errors."""
        if not self.kc:
            return {'success': False, 'errors': ['Kernel not available.'], 'output': '', 'code': code}
            
        msg_id = self.kc.execute(code)
        outputs = []
        errors = []

        while True:
            try:
                msg = self.kc.get_iopub_msg(timeout=5)
                msg_type = msg['header']['msg_type']
                content = msg['content']

                if msg['parent_header'].get('msg_id') != msg_id:
                    continue

                if msg_type == 'stream':
                    output_text = content['text'].strip()
                    if output_text:
                        outputs.append(output_text)
                        if 'ERROR:' in output_text or 'failed' in output_text.lower():
                            errors.append(output_text)
                elif msg_type == 'error':
                    errors.append('\n'.join(content['traceback']))
                elif msg_type == 'status' and content['execution_state'] == 'idle':
                    break
            except queue.Empty:
                break
        
        return {
            'success': len(errors) == 0,
            'errors': errors,
            'output': '\n'.join(outputs),
            'code': code
        }

    def shutdown(self):
        """Shuts down the kernel and cleans up the connection."""
        if self.km:
            print("Shutting down kernel...")
            self.kc.stop_channels()
            self.km.shutdown_kernel()
            print("Kernel shut down.")

# --- DomainSyntheticDataGenerator Class (v6) ---
class DomainSyntheticDataGenerator:
    """
    Generates synthetic "bad code" samples using multiple domain-aware
    heuristics.
    
    v6:
    - Implements parser that handles nested
      part definitions and instances. This will finally fix the
      EVSample.sysml bottleneck.
    """
    def __init__(self, validator):
        self.validator = validator
        self.heuristics = [
            # Heuristic 1: Mutate an existing valid connection
            self.mutate_valid_connection_to_domain_error,
            # Heuristic 2: Add a new invalid connection
            self.create_domain_mismatch_connection,
            # Heuristic 3: Swap unit incompatibly
            self.swap_unit_incompatibly,
            # Heuristic 4: Mismatch quantity kind type
            self.mismatch_quantity_kind_type,
            # Heuristic 5: Break compound unit expressions
            self.break_compound_unit_expression,
        ]
        # Caches to store file-specific parsing results
        self.part_def_cache = {}
        self.part_instance_cache = {}
        self.all_ports_cache = [] # (instance_name, port_name, full_path, domain, type)

    def _parse_file_definitions(self, code):
        """
        Parses all definitions and instances in the file and populates
        the caches. This is the main "parsing" step.
        """
        self.part_def_cache = {}
        self.part_instance_cache = {}
        self.all_ports_cache = []

        # 1. Find all Part Definitions (e.g., part def Battery { ... })
        part_def_regex = re.compile(r'part\s+def\s+([\w_]+)\s*\{(.+?)\}', re.DOTALL | re.IGNORECASE)
        port_regex = re.compile(r'port\s+([\w_]+)\s*:\s*([\w:]+)', re.IGNORECASE)
        for part_match in part_def_regex.finditer(code):
            part_name = part_match.group(1)
            part_body = part_match.group(2)
            ports = {}
            for port_match in port_regex.finditer(part_body):
                port_name = port_match.group(1)
                port_type = port_match.group(2).replace('~', '').strip()
                ports[port_name] = port_type
            self.part_def_cache[part_name] = ports

        # 2. Find all Part Instances (e.g., part battery: Battery { ... })
        # V6 - NEW ROBUST REGEX:
        # This regex finds `part` followed by an `instance_name`, `:`, `TypeName`,
        # and then either a `;` or a `{`.
        instance_regex = re.compile(
            r'part\s+'              # "part "
            r'([\w_]+)\s*:\s*'      # "instanceName : "
            r'([\w:]+)'             # "TypeName"
            r'\s*[\{;]',            # followed by either { or ;
            re.IGNORECASE
        )
        for match in instance_regex.finditer(code):
            instance_name, type_name = match.groups()
            self.part_instance_cache[instance_name] = type_name.replace('~', '').strip()

        # 3. Find all known ports in the file
        
        # Find SimpleVehicleModel-style ports (e.g., engine.drivePwrPort)
        for instance_name, part_type in self.part_instance_cache.items():
            part_ports = self.part_def_cache.get(part_type, {})
            for port_name, port_type in part_ports.items():
                domain = TYPE_TO_DOMAIN.get(port_type)
                if domain:
                    self.all_ports_cache.append((
                        instance_name, 
                        port_name, 
                        f"{instance_name}.{port_name}", 
                        domain, 
                        port_type
                    ))

        # Find EVSample-style ports (e.g., battery.batteryBehavior.output)
        ev_flow_regex = re.compile(r'flow\s+([\w\.]+)\s+to\s+([\w\.]+);', re.IGNORECASE)
        behavior_port_pattern = re.compile(r'\.(\w*behavior)\.(?:input|output)$', re.IGNORECASE)
        for match in ev_flow_regex.finditer(code):
            for full_path in [match.group(1), match.group(2)]:
                if behavior_port_pattern.search(full_path):
                    try:
                        instance_name, behavior, port_io = full_path.split('.')
                        part_type = self.part_instance_cache.get(instance_name)
                        if not part_type:
                            # This was the bug. The instance cache was empty.
                            # It should be fixed now, but we'll leave this check.
                            continue
                        
                        port_type_guess = f"{part_type.capitalize()}{port_io.capitalize()}"
                        domain = TYPE_TO_DOMAIN.get(port_type_guess)
                        if domain:
                            self.all_ports_cache.append((
                                instance_name,
                                port_io, # 'input' or 'output'
                                full_path,
                                domain,
                                port_type_guess
                            ))
                    except ValueError:
                        continue 

        self.all_ports_cache = list(set(self.all_ports_cache))
        # print(f"    - Found {len(self.all_ports_cache)} known ports.")

    def _get_line_number(self, code, match_start):
        """Convert a character position to a line number."""
        return code[:match_start].count('\n') + 1
        

    def _get_port_domain_from_path(self, full_path):
        """Gets the domain for a full port path (e.g., 'engine.drivePwrPort')."""
        for (inst, port, path, domain, p_type) in self.all_ports_cache:
            if path == full_path:
                return domain, p_type
        return None, None


    def mutate_valid_connection_to_domain_error(self, code):
        """
        Finds a valid connection and mutates one end
        to connect to an incompatible port.
        """
        mutated_variations = []
        if not self.all_ports_cache:
            return []

        connect_regex = re.compile(r'(connect|flow)\s+([\w\.]+)\s+to\s+([\w\.]+);', re.IGNORECASE)
        
        matches = list(connect_regex.finditer(code))
        
        for match in matches:
            statement_type, source_full, target_full = match.groups()

            source_domain, _ = self._get_port_domain_from_path(source_full)
            if not source_domain:
                continue 

            # Find an incompatible "victim" port
            for (victim_instance, victim_port, victim_full_path, victim_domain, victim_type) in self.all_ports_cache:
                
                if victim_domain not in VALID_CONNECTIONS.get(source_domain, []):
                    if target_full == victim_full_path:
                        continue # Don't mutate to itself

                    print(f"    - [H1] Found valid connection: {source_full} -> {target_full} (Domain: {source_domain})")
                    print(f"    - [H1] Found incompatible victim: {victim_full_path} (Domain: {victim_domain})")
                    
                    original_statement = match.group(0)
                    mutated_statement = f"{statement_type} {source_full} to {victim_full_path};"
                    
                    bad_code = code.replace(original_statement, mutated_statement, 1)
                    
                    line_num = self._get_line_number(code, match.start())
                    
                    error_explanation = (
                      f"ERROR:Domain violation: Port '{source_full}' ({source_domain}) cannot connect to '{victim_full_path}' ({victim_domain}) "
                      f"(line : {line_num})"
                    )
                    
                    mutated_variations.append((bad_code, error_explanation))
                       
        return mutated_variations


    def create_domain_mismatch_connection(self, code):
        """
        Adds a *new* connection between two incompatible ports
        found anywhere in the file.
        """
        mutated_variations = []
        if len(self.all_ports_cache) < 2:
            return []

        for i in range(len(self.all_ports_cache)):
            for j in range(i + 1, len(self.all_ports_cache)):
                (src_inst, src_port, src_full, src_domain, src_type) = self.all_ports_cache[i]
                (vic_inst, vic_port, vic_full, vic_domain, vic_type) = self.all_ports_cache[j]
                
                if vic_domain not in VALID_CONNECTIONS.get(src_domain, []):
                    print(f"    - [H2] Found incompatible pair: {src_full}({src_domain}) vs {vic_full}({vic_domain})")
                    
                    line_num = code.count('\n') + 2  # +2 because we're adding after the last line
                    
                    bad_code = code + f"\n\nconnect {src_full} to {vic_full};\n"
                    
                    error_explanation = (
                        f"ERROR:Domain violation: Port '{src_full}' ({src_domain}) cannot connect to '{vic_full}' ({vic_domain}) "
                        f"(line : {line_num})"
                    )
                    
                    mutated_variations.append((bad_code, error_explanation))
                       
        return mutated_variations
    
    
    def swap_unit_incompatibly(self, code):
        """
        Find attribute assignments with units and swap to incompatible units.
        Example: 'mass = 1800[kg]' -> 'mass = 1800[m]'
        """
        mutated_variations = []

        # Pattern: number[unit] where unit is in brackets
        # Matches: 1800[kg], 33['in'], 0.7[m], etc.
        unit_assignment_pattern = re.compile(
            r'=\s*([0-9.]+)\s*\[([^\]]+)\]',
            re.IGNORECASE
        )

        # Also need context: what attribute is this?
        # Pattern: attribute name :> ISQ::quantityKind ... = value[unit]
        attribute_context_pattern = re.compile(
            r'attribute\s+([\w_]+)\s*:>\s*(ISQ::[\w]+).*?=\s*([0-9.]+)\s*\[([^\]]+)\]',
            re.IGNORECASE | re.DOTALL
        )

        for match in attribute_context_pattern.finditer(code):
            attr_name = match.group(1)
            isq_type = match.group(2)
            value = match.group(3)
            unit = match.group(4)

            # Get quantity kind from ISQ type
            quantity_kind = quantity_kg.ISQ_TO_QUANTITY_KIND.get(isq_type)
            if not quantity_kind:
                continue

            # Get incompatible units for this quantity kind
            incompatible_units = quantity_kg.INCOMPATIBLE_UNITS.get(quantity_kind, [])
            if not incompatible_units:
                continue

            # Pick a random incompatible unit
            wrong_unit = random.choice(incompatible_units)

            # Create the mutation
            original_assignment = f"= {value}[{unit}]"
            mutated_assignment = f"= {value}[{wrong_unit}]"

            bad_code = code.replace(match.group(0), 
                                   match.group(0).replace(original_assignment, mutated_assignment), 
                                   1)
            
            line_num = self._get_line_number(code, match.start())

            error_explanation = (
                f"ERROR:Quantity mismatch: Attribute '{attr_name}' has type '{isq_type}' "
                f"({quantity_kind}) but assigned value with incompatible unit '{wrong_unit}' "
                f"(line : {line_num})"
            )

            mutated_variations.append((bad_code, error_explanation))

        return mutated_variations


    # --- HEURISTIC 4: Quantity Kind Type Mismatch ---
    def mismatch_quantity_kind_type(self, code):
        """
        Change the ISQ type on an attribute declaration to an incompatible type.
        Example: 'attribute mass :> ISQ::mass' -> 'attribute mass :> ISQ::length'
        """
        mutated_variations = []

        # Pattern: attribute name :> ISQ::type
        isq_attribute_pattern = re.compile(
            r'attribute\s+([\w_]+)\s*:>\s*(ISQ::[\w]+)',
            re.IGNORECASE
        )

        for match in isq_attribute_pattern.finditer(code):
            attr_name = match.group(1)
            isq_type = match.group(2)

            # Get alternative ISQ types
            alternatives = quantity_kg.ALTERNATIVE_ISQ_TYPES.get(isq_type, [])
            if not alternatives:
                continue

            # Pick a random alternative
            wrong_isq_type = random.choice(alternatives)

            # Create the mutation
            bad_code = code.replace(match.group(0),
                                   match.group(0).replace(isq_type, wrong_isq_type),
                                   1)

            line_num = self._get_line_number(code, match.start())
            
            error_explanation = (
                f"ERROR:Type mismatch: Attribute '{attr_name}' declared as '{wrong_isq_type}' "
                f"but should be '{isq_type}' based on semantic meaning "
                f"(line : {line_num})"
            )

            mutated_variations.append((bad_code, error_explanation))

        return mutated_variations


    # --- HEURISTIC 5: Break Compound Unit Expressions ---
    def break_compound_unit_expression(self, code):
        """
        Corrupt compound units by changing operators.
        Example: '[N*m]' -> '[N/m]' or '[kg⋅m²]' -> '[kg/m²]'
        """
        mutated_variations = []

        # Pattern: compound units with * or ⋅ operators
        compound_unit_pattern = re.compile(
            r'\[([A-Za-z\'Ω]+)\s*[\*⋅]\s*([A-Za-z0-9²³\'Ω]+)\]',
            re.IGNORECASE
        )

        for match in compound_unit_pattern.finditer(code):
            original_unit = match.group(0)
            part1 = match.group(1)
            part2 = match.group(2)

            # Create corrupted version by changing operator
            if '*' in original_unit or '⋅' in original_unit:
                corrupted_unit = f"[{part1}/{part2}]"
                corruption_desc = "multiplication to division"

            bad_code = code.replace(original_unit, corrupted_unit, 1)

            line_num = self._get_line_number(code, match.start())
            
            error_explanation = (
                f"ERROR:Unit expression corruption: Changed {corruption_desc} in compound unit "
                f"'{original_unit}' to '{corrupted_unit}' "
                f"(line : {line_num})"
            )

            mutated_variations.append((bad_code, error_explanation))

        return mutated_variations


    def generate_data_from_code(self, good_code, source_id):
        """
        Applies all heuristics and *ONLY* saves triplets where
        the validator SUCCEEDS, but our heuristic provides a
        domain error explanation.
        """
        training_data_records = []
        heuristic_stats = {}  # Track samples per heuristic
        
        try:
            self._parse_file_definitions(good_code)
            for port in self.all_ports_cache:
                print(f"      - {port}")
            
        except Exception as e:
            print(f"    ! Error parsing file definitions: {e}")
            return []

        for heuristic_func in self.heuristics:
            mutation_type = heuristic_func.__name__
            mutated_variations = []
            samples_added = 0

            try:
                mutated_variations = heuristic_func(good_code)

                if mutated_variations:
                    print(f"    - [{mutation_type}] Generated {len(mutated_variations)} candidate mutations")

                for bad_code, domain_explanation in mutated_variations:
                    result = self.validator.validate_code(bad_code)

                    if result['success']:
                        data_record = {
                            "source_id": source_id,
                            "mutation_type": mutation_type,
                            "bad_code": bad_code,
                            "error_message": domain_explanation, 
                            "good_code": good_code.strip()
                        }
                        training_data_records.append(data_record)
                        samples_added += 1
                    else:
                        print(f"    - Discarding mutation ({mutation_type}); it failed kernel validation.")
                        
                heuristic_stats[mutation_type] = samples_added
                if samples_added > 0:
                    print(f"    - [{mutation_type}] Accepted {samples_added} valid samples")

            except Exception as e:
                print(f"    ! Error in heuristic {mutation_type}: {e}")
                heuristic_stats[mutation_type] = 0
                continue

        unique_records = {r['bad_code']: r for r in training_data_records}.values()
        return list(unique_records)


if __name__ == "__main__":
    
   # base_manifest_dir = r"C:\Haitham\doctorals\other\incose\jupyter\vehicle_domain"
    base_manifest_dir = r"C:\Haitham\doctorals\other\incose\jupyter"
    manifest_json_path = r'C:\Haitham\doctorals\other\incose\jupyter\code\file_manifest_2.json'
    output_filename = 'synthetic_dataset_domain_aware_full12.jsonl'
    
    print("--- Starting DOMAIN-AWARE Synthetic Data Generation (v6) ---")

    # 1. Load the original manifest to get filename→source_id mapping
    print(f"\nStep 1a: Loading original manifest from '{manifest_json_path}'...")
    try:
        with open(manifest_json_path, 'r', encoding='utf-8') as f:
            original_manifest = json.load(f)
        
        # create full path to id lookup
        path_to_id = {}
        for file_path, source_id in original_manifest.items():
            normalized = os.path.normpath(file_path)
            if "-checkpoint" in normalized:
                continue
            path_to_id[normalized] = source_id
        
        print(f"Successfully loaded {len(path_to_id)} path mappings.")
        
        # DEBUG: Show a few manifest keys
        print(f"\nDEBUG: First 3 manifest keys:")
        for key in list(path_to_id.keys())[:3]:
            print(f"  '{key}'")
        
        # Track files for debugging
        files_found_on_disk = []
        files_found_and_matched = []
        files_found_but_not_matched = []
        
    except FileNotFoundError:
        print(f"Warning: Could not find '{manifest_json_path}'.")
        print("Continuing without source ID mapping...")
        path_to_id = {}
        files_found_on_disk = []
        files_found_and_matched = []
        files_found_but_not_matched = []

    # 1b. Scan for .sysml files in vehicle_domain
    print(f"\nStep 1b: Scanning for .sysml files in '{base_manifest_dir}'...")
    file_manifest = {}
    for root, _, files in os.walk(base_manifest_dir):
        # Skip vehicle_domain subdirectory to avoid duplicates
        if 'vehicle_domain' in root:
            continue
            
        for file in files:
            if file.endswith(".sysml") and "-checkpoint" not in file:
                file_path = os.path.join(root, file)
                normalized_path = os.path.normpath(file_path)
                
                # Track all files found on disk
                files_found_on_disk.append(normalized_path)
                
                # DEBUG: Show what we're looking for (only for CircularImport)
                if "CircularImport" in file:
                    print(f"\nDEBUG CircularImport:")
                    print(f"  Looking for: '{normalized_path}'")
                    print(f"  In manifest?: {normalized_path in path_to_id}")
                    # Show closest matches
                    matches = [k for k in path_to_id.keys() if "CircularImport" in k]
                    print(f"  Similar keys in manifest: {matches}")
                
                # Try to get source_id from original manifest
                if normalized_path in path_to_id:
                    source_id = path_to_id[normalized_path]
                    file_manifest[file_path] = source_id
                    files_found_and_matched.append(normalized_path)
                else:
                    files_found_but_not_matched.append(normalized_path)
                    print(f"  ! Warning: No source ID found for '{normalized_path}'. Skipping file.")
                    continue

    # Print debug summary
    print(f"\n=== DEBUG SUMMARY ===")
    print(f"Files in manifest: {len(path_to_id)}")
    print(f"Files found on disk: {len(files_found_on_disk)}")
    print(f"Files matched: {len(files_found_and_matched)}")
    print(f"Files found but not in manifest: {len(files_found_but_not_matched)}")
    
    manifest_files = set(path_to_id.keys())
    disk_files = set(files_found_on_disk)
    missing_from_disk = manifest_files - disk_files
    
    print(f"Files in manifest but NOT found on disk: {len(missing_from_disk)}")
    if missing_from_disk:
        print("First 10 missing files:")
        for f in list(missing_from_disk)[:10]:
            print(f"  {f}")

    if not file_manifest:
        print(f"Fatal: No .sysml files found in '{base_manifest_dir}'.")
        print("Please check the 'base_manifest_dir' path in this script.")
        exit()
        
    print(f"Successfully mapped {len(file_manifest)} files to process.")

    # 2. Initialize the SysML Validator
    print("\nStep 2: Initializing SysML Validator...")
    validator = SysMLValidator()
    
    if not validator.kc:
        print("Fatal: Could not start SysML kernel. Exiting.")
        exit()
        
    generator = DomainSyntheticDataGenerator(validator)

    # 3. Loop through manifest, generate data, and save
    print(f"\nStep 3: Generating data and saving to '{output_filename}'...")
    total_samples_generated = 0
    global_heuristic_stats = {
        "mutate_valid_connection_to_domain_error": 0,
        "create_domain_mismatch_connection": 0,
        "swap_unit_incompatibly": 0,
        "mismatch_quantity_kind_type": 0,
        "break_compound_unit_expression": 0,
    }
    
    try:
        with open(output_filename, 'w', encoding='utf-8') as f_out:
            
            for i, (file_path, source_id) in enumerate(file_manifest.items()):
                
                print(f"  Processing file {i+1}/{len(file_manifest)}: {source_id} ({os.path.basename(file_path)})")
                
                try:
                    with open(file_path, 'r', encoding='utf-8') as f_in:
                        good_code = f_in.read()
                except FileNotFoundError:
                    print(f"    ! File not found: {file_path}. Skipping.")
                    continue
                        
                if not good_code.strip():
                    print("    - Skipping empty file.")
                    continue
                
                synthetic_records = generator.generate_data_from_code(good_code, source_id)
                
                if synthetic_records:
                    print(f"    - Generated {len(synthetic_records)} valid domain-error samples.")
                    for record in synthetic_records:
                        mutation_type = record['mutation_type']
                        if mutation_type in global_heuristic_stats:
                            global_heuristic_stats[mutation_type] += 1
                    
                    for record in synthetic_records:
                        f_out.write(json.dumps(record) + '\n')
                    total_samples_generated += len(synthetic_records)
                else:
                    print("    - No valid (but wrong) domain samples generated for this file.")

    finally:
        # 4. Shutdown the kernel
        print("\nStep 4: Shutting down kernel...")
        validator.shutdown()
        print("\n" + "="*80)
        print(f"DOMAIN-AWARE data generation complete.")
        print(f"Total valid training samples generated: {total_samples_generated}")
        print(f"\nBreakdown by heuristic:")
        for heuristic, count in sorted(global_heuristic_stats.items(), key=lambda x: x[1], reverse=True):
            percentage = (count / total_samples_generated * 100) if total_samples_generated > 0 else 0
            print(f"  {heuristic:45s}: {count:4d} samples ({percentage:5.1f}%)")
        
        print(f"Dataset saved to: {output_filename}")
        print("="*80)

Successfully imported Vehicle Seed KG (v3).
Successfully imported Quantity KG (v1).
--- Starting DOMAIN-AWARE Synthetic Data Generation (v6) ---

Step 1a: Loading original manifest from 'C:\Haitham\doctorals\other\incose\jupyter\code\file_manifest_2.json'...
Successfully loaded 1286 path mappings.

DEBUG: First 3 manifest keys:
  'C:\Haitham\doctorals\other\incose\jupyter\data2\SE_Models\DroneModelLogical.sysml'
  'C:\Haitham\doctorals\other\incose\jupyter\data2\SE_Models\Drone_BaseArchitecture.sysml'
  'C:\Haitham\doctorals\other\incose\jupyter\data2\SE_Models\EIT_System_Use_Cases.sysml'

Step 1b: Scanning for .sysml files in 'C:\Haitham\doctorals\other\incose\jupyter'...

DEBUG CircularImport:
  Looking for: 'C:\Haitham\doctorals\other\incose\jupyter\examples\Import Tests\CircularImport.sysml'
  In manifest?: True
  Similar keys in manifest: ['C:\\Haitham\\doctorals\\other\\incose\\jupyter\\examples\\Import Tests\\CircularImport.sysml']

=== DEBUG SUMMARY ===
Files in manifest: 1286


In [3]:
!jupyter nbconvert --to python vehicle_kg.ipynb

[NbConvertApp] Converting notebook vehicle_kg.ipynb to python
[NbConvertApp] Writing 1548 bytes to vehicle_kg.py
